In [13]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm


In [14]:
folders = [
# r"../Data/raw/saxophone_data",
# r"../Data/raw/piano_data",
# r"../Data/raw/guitar_data",
r"../Data/raw/symphony_data",
# r"../Data/raw/violin_data"
]

In [15]:
def extract_features(y, sr):
    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    # Spectral features
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

    # ZCR
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))

    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    # Xử lý tương thích: librosa bản mới trả về tempo là mảng 1D thay vì số float
    tempo_val = tempo[0] if isinstance(tempo, np.ndarray) else tempo

    features = np.hstack([
        mfcc_mean,
        chroma_mean,
        centroid,
        bandwidth,
        rolloff,
        zcr,
        tempo_val
    ])

    return features

In [16]:
data = []
# --- CẤU HÌNH SLIDING WINDOW ---
window_duration = 5.0  # Độ dài mỗi cửa sổ là 5 giây
overlap_ratio = 0.5    # Chồng lấp 50%
hop_duration = window_duration * (1 - overlap_ratio) # Bước nhảy là 2.5 giây

for folder in folders:
    instrument = os.path.basename(folder)

    for file in tqdm(os.listdir(folder)):
        if file.endswith(".mp3"):
            path = os.path.join(folder, file)

            try:
                # 1. Đọc file âm thanh
                y, sr = librosa.load(path, sr=22050)
                
                # 2. Quy đổi thời gian ra số lượng sample (mẫu)
                window_samples = int(window_duration * sr)
                hop_samples = int(hop_duration * sr)

                # 3. Logic Sliding Window
                start_sample = 0
                segment_id = 1
                
                # Bỏ qua các file có tổng thời lượng ngắn hơn 1 cửa sổ
                if len(y) < window_samples:
                    print(f"Bỏ qua {file} vì thời lượng quá ngắn.")
                    continue

                # Trượt cửa sổ cho đến khi cửa sổ vượt quá chiều dài của file
                while start_sample + window_samples <= len(y):
                    end_sample = start_sample + window_samples
                    y_segment = y[start_sample:end_sample]

                    # Trích xuất đặc trưng trên đoạn cửa sổ hiện tại
                    features = extract_features(y_segment, sr)

                    # Ghi nhận dữ liệu
                    row = {
                        "file_name": file,
                        "path": path,
                        "instrument": instrument,
                        "segment_id": segment_id
                    }

                    # MFCC (1-13)
                    for i in range(13):
                        row[f"mfcc_{i+1}"] = features[i]

                    # Chroma
                    chroma_labels = [
                        "C","Csharp","D","Dsharp","E","F",
                        "Fsharp","G","Gsharp","A","Asharp","B"
                    ]
                    for i, label in enumerate(chroma_labels):
                        row[f"chroma_{label}"] = features[13 + i]

                    # Các đặc trưng khác
                    row["spectral_centroid"] = features[25]
                    row["spectral_bandwidth"] = features[26]
                    row["spectral_rolloff"] = features[27]
                    row["zero_crossing_rate"] = features[28]
                    row["tempo"] = features[29]

                    data.append(row)
                    
                    # Tiến cửa sổ lên một bước nhảy (hop)
                    start_sample += hop_samples
                    segment_id += 1

            except Exception as e:
                print(f"Error processing {file}: {e}")

df = pd.DataFrame(data)

 95%|█████████▍| 122/129 [04:46<00:16,  2.34s/it]e:\Anaconda\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
100%|██████████| 129/129 [05:02<00:00,  2.35s/it]


In [17]:
df

,file_name,path,instrument,segment_id,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,...,chroma_G,chroma_Gsharp,chroma_A,chroma_Asharp,chroma_B,spectral_centroid,spectral_bandwidth,spectral_rolloff,zero_crossing_rate,tempo
0,sym_001_classic.mp3,../Data/raw/symphony_data\sym_001_classic.mp3,symphony_data,1,-439.986053,85.139694,0.458754,16.637480,4.715432,7.667087,...,0.787704,0.598356,0.572521,0.570786,0.561459,1840.292918,2323.627972,4267.761230,0.061315,117.453835
1,sym_001_classic.mp3,../Data/raw/symphony_data\sym_001_classic.mp3,symphony_data,2,-212.896347,140.201080,-11.932239,22.817383,0.743505,8.239263,...,0.811418,0.425508,0.339081,0.358247,0.432731,1489.021309,1842.230906,3087.073771,0.071162,143.554688
2,sym_001_classic.mp3,../Data/raw/symphony_data\sym_001_classic.mp3,symphony_data,3,-184.397034,132.663391,-6.675671,25.261106,0.345872,10.353242,...,0.703835,0.398058,0.423456,0.294041,0.467659,1612.430756,1991.305824,3407.130941,0.069897,143.554688
3,sym_001_classic.mp3,../Data/raw/symphony_data\sym_001_classic.mp3,symphony_data,4,-174.846100,127.903214,-6.703556,27.096706,-1.176579,11.535839,...,0.662742,0.474079,0.477346,0.299638,0.399277,1657.197184,2052.079829,3507.768758,0.071653,143.554688
4,sym_001_classic.mp3,../Data/raw/symphony_data\sym_001_classic.mp3,symphony_data,5,-179.530823,128.460129,-8.316603,27.567307,-0.174197,12.435596,...,0.690341,0.486605,0.429626,0.277535,0.376197,1651.611849,2033.743312,3498.846436,0.072573,143.554688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1414,sym_129_classic.mp3,../Data/raw/symphony_data\sym_129_classic.mp3,symphony_data,7,-516.179565,134.790405,-6.459749,9.287603,-0.186479,11.679767,...,0.217816,0.259035,0.065726,0.223434,0.577339,1184.215321,1709.357973,1925.875854,0.052743,143.554688
1415,sym_129_classic.mp3,../Data/raw/symphony_data\sym_129_classic.mp3,symphony_data,8,-510.578979,151.344818,-5.269817,11.355229,0.095805,6.550085,...,0.180965,0.274725,0.065368,0.213705,0.563616,1057.760645,1589.487163,1733.472697,0.044495,172.265625
1416,sym_129_classic.mp3,../Data/raw/symphony_data\sym_129_classic.mp3,symphony_data,9,-503.638947,139.103470,-14.552151,0.085143,-3.697760,3.094876,...,0.179944,0.396874,0.063420,0.186297,0.523875,1185.708696,1538.681623,1801.561483,0.064514,172.265625
1417,sym_129_classic.mp3,../Data/raw/symphony_data\sym_129_classic.mp3,symphony_data,10,-487.227142,130.325500,-14.470521,-5.013844,-8.025957,3.377542,...,0.162297,0.635024,0.089506,0.120764,0.637255,1316.558728,1511.950149,1932.654826,0.089394,123.046875


In [18]:

df.to_csv("../Results/symphony_feature_database.csv", index=False)

print("Feature extraction finished!")
print("Total files processed:", len(df))

Feature extraction finished!
Total files processed: 1419
